In [1]:
import pandas as pd

## 1. Hybridization control normalization

In [2]:
TYPE_HCE = "Hybridization Control Elution"

In [3]:
features = pd.read_csv(
    "/mnt/data/processed/features.csv"
)
samples = pd.read_csv(
    "/mnt/data/processed/samples.csv"
)
measurements_raw = pd.read_csv(
    "/mnt/data/processed/measurements.raw.csv"
)

In [4]:
features_hce = features.loc[
    features["Type"] == TYPE_HCE
]

In [5]:
measurements_hce = (
    features_hce["ProbeId"]
                .to_frame()
                .join(
                    measurements_raw.set_index("ProbeId"),
                    on = "ProbeId"
                )
)

In [6]:
measurements_hce_ref = (
    measurements_hce.groupby(["PlateId", "ProbeId"])
                     .value
                     .median()
                     .rename("value_ref")
)

In [7]:
measurements_hce_scale_factor = measurements_hce.join(
    measurements_hce_ref,
    on = ["PlateId", "ProbeId"]
)
measurements_hce_scale_factor["value_scale_factor"] = (
    measurements_hce_scale_factor.value_ref /
       measurements_hce_scale_factor.value
)
measurements_hce_scale_factor = (
    measurements_hce_scale_factor.groupby(["PlateId", "PlatePosition"])
                                 .value_scale_factor
                                 .median()
)

In [8]:
measurements_hcn = measurements_raw.join(
    measurements_hce_scale_factor,
    on = ["PlateId", "PlatePosition"]
)
measurements_hcn.value *= measurements_hcn.value_scale_factor
measurements_hcn = measurements_hcn.drop("value_scale_factor", axis = 1)
measurements_hcn.to_csv(
    "/mnt/data/processed/measurements.hybridization_control_normalized.csv",
    index=False
)

measurements_hcn

,PlateId,PlatePosition,ProbeId,value
0,P0031168,A1,10000-28,628.862531
1,P0031168,A10,10000-28,621.353074
2,P0031168,A11,10000-28,550.560857
3,P0031168,A12,10000-28,557.348126
4,P0031168,A2,10000-28,394.584597
...,...,...,...,...
15571795,P0031201,H5,9999-1,2903.102446
15571796,P0031201,H6,9999-1,1622.180841
15571797,P0031201,H7,9999-1,7548.129488
15571798,P0031201,H8,9999-1,6344.403187


In [9]:
import numpy as np

measurements_hcn[
    np.logical_and(
        np.logical_and(
            measurements_hcn.PlateId == "P0031168",
            measurements_hcn.PlatePosition == "F7"
        ),
        measurements_hcn.ProbeId.isin(["2171-12", "2178-55"])
    )
]

,PlateId,PlatePosition,ProbeId,value
6883969,P0031168,F7,2171-12,131183.105962
6986469,P0031168,F7,2178-55,20970.845615


## 2. Median signal normalization on calibrators

In [10]:
TYPE_CALIBRATOR = "Calibrator"

In [11]:
features_hce = features.loc[
    features["Type"] == TYPE_HCE
]

In [12]:
samples_calibrators = samples.loc[
    samples["SampleType"] == TYPE_CALIBRATOR
]

In [13]:
measurements_calibrators = (
    samples_calibrators.loc[:, ["PlateId", "PlatePosition"]]
                       .join(
                           measurements_hcn.set_index(["PlateId", "PlatePosition"]),
                           on = ["PlateId", "PlatePosition"]
                       )
)
measurements_calibrators = measurements_calibrators[~measurements_calibrators.ProbeId.isin(features_hce.ProbeId)]

In [14]:
measurements_calibrators_ref = (
    measurements_calibrators.groupby(["PlateId", "ProbeId"])
                            .value
                            .median()
                            .rename("value_ref")
)

In [15]:
measurements_calibrators_scale_factor = measurements_calibrators.join(
    measurements_calibrators_ref,
    on = ["PlateId", "ProbeId"]
)
measurements_calibrators_scale_factor["value_ratio"] = (
    measurements_calibrators_scale_factor.value / 
        measurements_calibrators_scale_factor.value_ref
)
measurements_calibrators_scale_factor = ( 
    1 / measurements_calibrators_scale_factor.join(
                                                 features.set_index("ProbeId")["Dilution"],
                                                 on = "ProbeId"
                                              ).groupby(["PlateId", "PlatePosition", "Dilution"])
                                              .value_ratio
                                              .median()
                                              .rename("value_scale_factor")
)

In [16]:
measurements_msncal = (
    measurements_hcn.join(
                        features.set_index("ProbeId")["Dilution"],
                        on = "ProbeId"
                     ).join(
                         measurements_calibrators_scale_factor,
                         on = ["PlateId", "PlatePosition", "Dilution"]
                     )
)
measurements_msncal.loc[~measurements_msncal["value_scale_factor"].isna(), "value"] *= (
    measurements_msncal.loc[~measurements_msncal["value_scale_factor"].isna(), "value_scale_factor"]
)
measurements_msncal = measurements_msncal.drop(["Dilution", "value_scale_factor"], axis = 1)
measurements_msncal.to_csv(
    "/mnt/data/processed/measurements.median_signal_normalized_calibrators.csv",
    index=False
)

measurements_msncal

,PlateId,PlatePosition,ProbeId,value
0,P0031168,A1,10000-28,628.862531
1,P0031168,A10,10000-28,621.353074
2,P0031168,A11,10000-28,550.560857
3,P0031168,A12,10000-28,557.348126
4,P0031168,A2,10000-28,394.584597
...,...,...,...,...
15571795,P0031201,H5,9999-1,2903.102446
15571796,P0031201,H6,9999-1,1622.180841
15571797,P0031201,H7,9999-1,7548.129488
15571798,P0031201,H8,9999-1,6344.403187


## 3. Plate-scale normalization

In [17]:
TYPE_CALIBRATOR = "Calibrator"

In [18]:
samples_calibrators = samples.loc[
    samples["SampleType"] == TYPE_CALIBRATOR
]

In [19]:
measurements_calibrators = (
    samples_calibrators.loc[:, ["PlateId", "PlatePosition"]]
                       .join(
                           measurements_msncal.set_index(["PlateId", "PlatePosition"]),
                           on = ["PlateId", "PlatePosition"]
                       )
)

In [20]:
measurements_psn_ref = (
    measurements_calibrators.groupby(["ProbeId"])
                            .value
                            .median()
                            .rename("value_ref")
)

In [21]:
measurements_psn_scale_factor = (
    measurements_calibrators.groupby(["PlateId", "ProbeId"])
                            .value
                            .median()
                            .to_frame()
                            .reset_index()
                            .join(
                                measurements_psn_ref,
                                on = ["ProbeId"]
                            )
)
measurements_psn_scale_factor["value_scale_factor"] = (
    measurements_psn_scale_factor.value_ref /
       measurements_psn_scale_factor.value
)
measurements_psn_scale_factor = (
    measurements_psn_scale_factor.groupby(["PlateId"])
                                 .value_scale_factor
                                 .median()
)

In [22]:
measurements_psn = (
    measurements_msncal.join(
                           measurements_psn_scale_factor,
                           on = ["PlateId"]
                       )
)
measurements_psn.value *= measurements_psn.value_scale_factor
measurements_psn = measurements_psn.drop(["value_scale_factor"], axis = 1)
measurements_psn.to_csv(
    "/mnt/data/processed/measurements.plate_scale_normalized.csv",
    index=False
)

measurements_psn

,PlateId,PlatePosition,ProbeId,value
0,P0031168,A1,10000-28,661.315406
1,P0031168,A10,10000-28,653.418418
2,P0031168,A11,10000-28,578.972921
3,P0031168,A12,10000-28,586.110452
4,P0031168,A2,10000-28,414.947401
...,...,...,...,...
15571795,P0031201,H5,9999-1,2586.745837
15571796,P0031201,H6,9999-1,1445.408702
15571797,P0031201,H7,9999-1,6725.595425
15571798,P0031201,H8,9999-1,5653.041474


## 4. Inter-plate calibration

In [23]:
TYPE_CALIBRATOR = "Calibrator"

In [24]:
samples_calibrators = samples.loc[
    samples["SampleType"] == TYPE_CALIBRATOR
]

In [25]:
measurements_calibrators = (
    samples_calibrators.loc[:, ["PlateId", "PlatePosition"]]
                       .join(
                           measurements_psn.set_index(["PlateId", "PlatePosition"]),
                           on = ["PlateId", "PlatePosition"]
                       )
)

In [26]:
measurements_interplate_ref = (
    measurements_calibrators.groupby(["ProbeId"])
                            .value
                            .median()
                            .rename("value_ref")
)

In [28]:
measurements_interplate_scale_factor = (
    measurements_calibrators.groupby(["PlateId", "ProbeId"])
                            .value
                            .median()
                            .to_frame()
                            .reset_index()
                            .join(
                                measurements_interplate_ref,
                                on = ["ProbeId"]
                            )
)
measurements_interplate_scale_factor["value_scale_factor"] = (
    measurements_interplate_scale_factor.value_ref /
       measurements_interplate_scale_factor.value
)
measurements_interplate_scale_factor = (
    measurements_interplate_scale_factor.set_index(["PlateId", "ProbeId"])
                                        .value_scale_factor
)

In [30]:
measurements_ipn = (
    measurements_psn.join(
                        measurements_interplate_scale_factor,
                        on = ["PlateId", "ProbeId"]
                    )
)
measurements_ipn.value *= measurements_ipn.value_scale_factor
measurements_ipn = measurements_ipn.drop(["value_scale_factor"], axis = 1)
measurements_ipn.to_csv(
    "/mnt/data/processed/measurements.interplate_normalized.csv",
    index=False
)

measurements_ipn

,PlateId,PlatePosition,ProbeId,value
0,P0031168,A1,10000-28,681.308251
1,P0031168,A10,10000-28,673.172522
2,P0031168,A11,10000-28,596.476394
3,P0031168,A12,10000-28,603.829706
4,P0031168,A2,10000-28,427.492065
...,...,...,...,...
15571795,P0031201,H5,9999-1,2416.677534
15571796,P0031201,H6,9999-1,1350.378799
15571797,P0031201,H7,9999-1,6283.414140
15571798,P0031201,H8,9999-1,5281.376367


## 5. Median signal normalization on all sample types